### # Data Profiling – Vouchers

This notebook profiles the vouchers dataset to understand discount types,
validity periods, and data quality.


In [0]:
#reading data from volume
df = (
    spark.read
    .option("header", True)
    .csv("/Volumes/workspace/default/coffee_raw_volume/vouchers/")
)

df.printSchema()
df.show()


In [0]:
# Row count & uniqueness
from pyspark.sql.functions import count, countDistinct

df.select(
    count("*").alias("total_vouchers"),
    countDistinct("voucher_id").alias("distinct_voucher_ids"),
    countDistinct("voucher_code").alias("distinct_voucher_codes")
).show()


In [0]:
# Discount type distribution
df.groupBy("discount_type").count().show()


In [0]:
#validity period check
from pyspark.sql.functions import min, max

df.select(
    min("valid_from").alias("min_valid_from"),
    max("valid_to").alias("max_valid_to")
).show()


In [0]:
# Null checks
df.selectExpr(
    "sum(case when voucher_id is null then 1 else 0 end) as null_voucher_id",
    "sum(case when voucher_code is null then 1 else 0 end) as null_voucher_code",
    "sum(case when discount_type is null then 1 else 0 end) as null_discount_type",
    "sum(case when discount_value is null then 1 else 0 end) as null_discount_value",
    "sum(case when valid_from is null then 1 else 0 end) as null_valid_from",
    "sum(case when valid_to is null then 1 else 0 end) as null_valid_to"
).show()


###  Key Observations

- The vouchers dataset contains 16 records with unique voucher_id values.
- voucher_code contains 8 unique values, indicating that some voucher codes have multiple validity periods.
- Discount types include percentage-based and fixed-amount discounts, with percentage discounts being more common.
- Discount values fall within reasonable ranges.
- Vouchers are time-bound with defined valid_from and valid_to periods.

